# Autokeras

AutoKeras is an AutoML (Automated Machine Learning) library built on TensorFlow and Keras, designed to facilitate the creation and optimization of deep learning models. Its main goal is to automate the process of searching for neural network architectures, eliminating the need for manual intervention in choosing hyperparameters and network layers.

python 3.9.20
tensorflow==2.10.1
autokeras==1.1.0
numpy==1.26.4
keras-nlp==0.10.0

In [1]:
import pandas as pd
import numpy as np
import sys
import io
import random

# Machine Learning imports
import autokeras as ak
import tensorflow as tf
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, hamming_loss

# Configure stdout to UTF-8
try:
    sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8')
except Exception as e:
    print(f"Could not set UTF-8 encoding: {e}")

# -----------------------------------------------------------------
# Seed Fixing for Reproducibility
# -----------------------------------------------------------------
def set_seed(seed_value=42):
    """Sets random seeds for reproducibility."""
    np.random.seed(seed_value)
    random.seed(seed_value)
    tf.random.set_seed(seed_value)  # Important for AutoKeras / TensorFlow
    
    # Try to enforce TensorFlow determinism
    try:
        # For TF 2.8+
        tf.config.experimental.enable_op_determinism()
    except AttributeError:
        try:
            # For older TF versions
            tf.random.set_seed(seed_value)
        except Exception as e2:
            print(f"Warning: Could not set TensorFlow seed: {e2}")

set_seed(42)
print("Random seeds (random, numpy, tensorflow) fixed to 42.")

Using TensorFlow backend


c:\Users\FUNPEC\anaconda3\envs\automl\lib\site-packages\tensorflow_hub\__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
c:\Users\FUNPEC\anaconda3\envs\automl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Could not set UTF-8 encoding: 'OutStream' object has no attribute 'buffer'
Random seeds (random, numpy, tensorflow) fixed to 42.


In [2]:
# -----------------------------------------------------------------
# 2. Load and Aggregate the Data (Multi-Label Logic)
# -----------------------------------------------------------------
print("Loading and processing the dataset...")
try:
    df = pd.read_csv("dataset.csv")
except FileNotFoundError:
    print("Error: File 'dataset.csv' not found.")
    sys.exit()

df = df.dropna(subset=['courseName', 'comp_name', 'courseDescription'])

# Aggregate by course FIRST
print("Grouping by course...")
df_agg = df.groupby('courseName', as_index=False).agg({
    'courseDescription': 'first',
    'comp_name': lambda x: list(set(x))
})

print(f"Number of courses (aggregated) before filtering: {len(df_agg)}")

# -----------------------------------------------------------------
# 3. CORRECT FILTERING LOGIC (Post-Aggregation)
# -----------------------------------------------------------------
competency_counts = df_agg['comp_name'].explode().value_counts()

# Using a threshold of 5 to ensure non-zero 'support' in the test set
MIN_COURSE_COUNT = 5
rare_competencies = competency_counts[competency_counts < MIN_COURSE_COUNT].index
print(f"Identified {len(rare_competencies)} rare competencies (in < {MIN_COURSE_COUNT} courses).")

df_agg['comp_name_filtered'] = df_agg['comp_name'].apply(
    lambda comp_list: [c for c in comp_list if c not in rare_competencies]
)
df_filtered = df_agg[df_agg['comp_name_filtered'].apply(len) > 0].copy()
print(f"Number of courses after filtering: {len(df_filtered)}")

# -----------------------------------------------------------------
# 4. Prepare X (Texts) and y (Labels)
# -----------------------------------------------------------------
# AutoKeras uses raw text as X
df_filtered['combinedText'] = df_filtered['courseDescription'].fillna('')
texts = np.array(df_filtered['combinedText'].astype(str).tolist(), dtype=str).reshape(-1,)

# y (target) is the multi-label matrix
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df_filtered['comp_name_filtered'])
num_labels = len(mlb.classes_)

print(f"X shape (Texts): {texts.shape}")
print(f"y shape (Filtered): {y.shape}")
print(f"Number of classes (competencies) after filtering: {len(mlb.classes_)}")

# -----------------------------------------------------------------
# 5. Split the Data
# -----------------------------------------------------------------
x_train, x_test, y_train, y_test = train_test_split(
    texts, y, test_size=0.2, random_state=42
)
print(f"Split completed: {len(x_train)} training samples, {len(x_test)} test samples.")

Loading and processing the dataset...
Grouping by course...
Number of courses (aggregated) before filtering: 341
Identified 14 rare competencies (in < 5 courses).
Number of courses after filtering: 337
X shape (Texts): (337,)
y shape (Filtered): (337, 53)
Number of classes (competencies) after filtering: 53
Split completed: 269 training samples, 68 test samples.


In [3]:
# -----------------------------------------------------------------
# 6. Train and Evaluate (AutoKeras)
# -----------------------------------------------------------------
print("\n--- Training AutoKeras TextClassifier ---")

# TensorFlow seeds were set in Cell 1
clf = ak.TextClassifier(
    max_trials=3,  # (As in the user's script)
    objective='val_loss',  # Optimizing loss (BCE) is a fair default
    multi_label=True,
    overwrite=True,
    project_name='autokeras_fair_comparison_v2',
    seed=42  # Set AutoKeras seed as well
)

# Train the model (AutoKeras uses 10% for validation by default)
clf.fit(x_train, y_train, epochs=10, batch_size=32)
print("AutoKeras training completed.")

# -----------------------------------------------------------------
# 7. Evaluation (with Threshold Optimization)
# -----------------------------------------------------------------
print(f"\n--- Optimizing Thresholds for AutoKeras ---")

# .predict() in multi-label AutoKeras returns probabilities
y_proba = clf.predict(x_test)

# --- Threshold Optimization Loop ---
thresholds = np.arange(0.05, 1.0, 0.05)
best_thresholds = []

for i in range(y_test.shape[1]):
    y_true_col = y_test[:, i]
    y_proba_col = y_proba[:, i]
    
    best_f1 = 0
    best_thresh = 0.5
    for thresh in thresholds:
        y_pred_col = (y_proba_col > thresh).astype(int)
        score = f1_score(y_true_col, y_pred_col, average='binary', zero_division=0)
        
        if score > best_f1:
            best_f1 = score
            best_thresh = thresh
            
    best_thresholds.append(best_thresh)

y_pred_optimized = np.zeros(y_test.shape)
for i in range(y_test.shape[1]):
    y_pred_optimized[:, i] = (y_proba[:, i] > best_thresholds[i]).astype(int)

# --- Show results ---
print(f"\n--- AutoKeras Evaluation (OPTIMIZED) ---")
print(f"Hamming Loss: {hamming_loss(y_test, y_pred_optimized):.4f}")
print(f"F1 Score (micro): {f1_score(y_test, y_pred_optimized, average='micro'):.4f}")
print(f"F1 Score (macro): {f1_score(y_test, y_pred_optimized, average='macro', zero_division=0):.4f}")
print(f"Classification Report (Optimized AutoKeras):\n")
print(classification_report(y_test, y_pred_optimized, target_names=mlb.classes_, zero_division=0))

# -----------------------------------------------------------------
# SNIPPET: Partial Hit Calculation
# -----------------------------------------------------------------
y_pred_model = y_pred_optimized
y_test_true = y_test
total_test_samples = len(y_test_true)
positive_hits = y_test_true * y_pred_model
hits_sum_per_row = positive_hits.sum(axis=1)
rows_with_at_least_one_hit = (hits_sum_per_row > 0).sum()
percentage = (rows_with_at_least_one_hit / total_test_samples) * 100

print(f"\n--- Partial Hit Metric (At Least 1) ---")
print(f"Rows with at least 1 hit: {rows_with_at_least_one_hit} out of {total_test_samples}")
print(f"Partial Hit Percentage: {percentage:.2f}%")

Trial 3 Complete [00h 00m 01s]

Best val_loss So Far: 0.17520996928215027
Total elapsed time: 00h 00m 06s
Epoch 1/10
9/9 [==============================] - 0s 24ms/step - loss: 0.6509 - accuracy: 0.0706
Epoch 2/10
9/9 [==============================] - 0s 22ms/step - loss: 0.4464 - accuracy: 0.0558
Epoch 3/10
9/9 [==============================] - 0s 26ms/step - loss: 0.2387 - accuracy: 0.0781
Epoch 4/10
9/9 [==============================] - 0s 24ms/step - loss: 0.2284 - accuracy: 0.1375
Epoch 5/10
9/9 [==============================] - 0s 26ms/step - loss: 0.2070 - accuracy: 0.0967
Epoch 6/10
9/9 [==============================] - 0s 24ms/step - loss: 0.2035 - accuracy: 0.1190
Epoch 7/10
9/9 [==============================] - 0s 25ms/step - loss: 0.1972 - accuracy: 0.1301
Epoch 8/10
9/9 [==============================] - 0s 25ms/step - loss: 0.1962 - accuracy: 0.1375
Epoch 9/10
9/9 [==============================] - 0s 25ms/step - loss: 0.1952 - accuracy: 0.1264
Epoch 10/10
9/9 [====

INFO:tensorflow:Assets written to: .\autokeras_fair_comparison_v2\best_model\assets


INFO:tensorflow:Assets written to: .\autokeras_fair_comparison_v2\best_model\assets


AutoKeras training completed.

--- Optimizing Thresholds for AutoKeras ---
3/3 [==============================] - 0s 5ms/step

--- AutoKeras Evaluation (OPTIMIZED) ---
Hamming Loss: 0.0622
F1 Score (micro): 0.1515
F1 Score (macro): 0.0086
Classification Report (Optimized AutoKeras):

                                                     precision    recall  f1-score   support

                Acessibilidade e inclusão - Docente       0.00      0.00      0.00        10
                              Análise de evidências       0.00      0.00      0.00         1
                               Análise de problemas       0.00      0.00      0.00         0
                         Aprendizagem autorregulada       0.00      0.00      0.00         7
                          Aprendizagem colaborativa       0.00      0.00      0.00         7
                           Colaboração profissional       0.00      0.00      0.00         2
                                        Comunicação       0.00 

In [4]:
# --- Top-K Evaluation Function ---
def evaluate_top_k(y_test, y_proba, k_values=[1, 3, 5, 7, 10]):
    """
    Evaluates the model by forcing K predictions per sample based on the highest probabilities.
    """
    print(f"\n{'='*60}\n DETAILED RESULTS BY TOP-K \n{'='*60}")
    
    total_samples = y_test.shape[0]
    
    # Ensure y_proba is a float numpy array (AutoKeras may return non-numeric types if not handled)
    try:
        y_proba = np.array(y_proba, dtype=float)
    except Exception:
        print("WARNING: y_proba does not appear to be numeric. Top-K evaluation may fail.")

    for k in k_values:
        print(f"\n>>> TOP-{k} ANALYSIS (Forcing the {k} highest probabilities) <<<")
        
        # 1. Build the Top-K prediction matrix
        # Create a zeros matrix with the same shape as y_test
        y_pred_k = np.zeros_like(y_test)
        
        # Get the indices of the K highest probabilities for each row
        # argsort sorts from lowest to highest, so we take the last k ([-k:])
        top_k_indices = np.argsort(y_proba, axis=1)[:, -k:]
        
        # Fill with 1 only at the Top-K indices
        for i in range(total_samples):
            y_pred_k[i, top_k_indices[i]] = 1
            
        # 2. Compute classic metrics (F1, Hamming) for this forced scenario
        f1_mic = f1_score(y_test, y_pred_k, average='micro')
        f1_mac = f1_score(y_test, y_pred_k, average='macro', zero_division=0)
        h_loss = hamming_loss(y_test, y_pred_k)
        
        # 3. Compute ranking metrics (Precision@K and Hit Rate)
        total_hits = 0
        samples_with_hit = 0
        
        for i in range(total_samples):
            true_indices = np.where(y_test[i] == 1)[0]  # True indices
            pred_indices = top_k_indices[i]             # Predicted indices (Top-K)
            
            # Intersection between ground truth and predictions
            hits = len(set(pred_indices) & set(true_indices))
            total_hits += hits
            
            if hits > 0:
                samples_with_hit += 1
        
        # Precision@K: Of the K guesses we made, how many were correct (on average)?
        precision_at_k = total_hits / (total_samples * k)
        
        # Hit Rate (Partial Hit): In how many samples did we get at least 1 correct label?
        hit_rate_at_k = (samples_with_hit / total_samples) * 100
        
        print(f"{'-'*40}")
        print(f"Hamming Loss:        {h_loss:.4f}")
        print(f"F1 Score (Micro):    {f1_mic:.4f}")
        print(f"F1 Score (Macro):    {f1_mac:.4f}")
        print(f"{'-'*40}")
        print(f"Precision@{k}:        {precision_at_k:.4f} (Average hits within the {k})")
        print(f"Hit Rate@{k}:         {hit_rate_at_k:.2f}% (Samples with at least 1 hit)")
        print(f"{'-'*40}")

# =================================================================
# UPDATE TO STEPS 7 AND 8 IN YOUR SCRIPT
# =================================================================

print("\n--- Generating Probabilities for the Test Set ---")
# AutoKeras predict returns numeric arrays for multi-label regression/classification
y_proba = clf.predict(x_test)

# 1. Run your threshold optimization (kept from your original code)
print(f"\n--- Evaluation 1: Threshold Optimization ---")
# ... (Keep your threshold loop here to compute 'y_pred_optimized') ...
# (Omitted here for brevity, but keep it if you want the optimized F1 results)

# 2. Run the new Top-K evaluation
evaluate_top_k(y_test, y_proba, k_values=[1, 3, 5, 7, 10])


--- Generating Probabilities for the Test Set ---
3/3 [==============================] - 0s 4ms/step

--- Evaluation 1: Threshold Optimization ---

 DETAILED RESULTS BY TOP-K 

>>> TOP-1 ANALYSIS (Forcing the 1 highest probabilities) <<<
----------------------------------------
Hamming Loss:        0.0622
F1 Score (Micro):    0.1515
F1 Score (Macro):    0.0086
----------------------------------------
Precision@1:        0.2941 (Average hits within the 1)
Hit Rate@1:         29.41% (Samples with at least 1 hit)
----------------------------------------

>>> TOP-3 ANALYSIS (Forcing the 3 highest probabilities) <<<
----------------------------------------
Hamming Loss:        0.0954
F1 Score (Micro):    0.1400
F1 Score (Macro):    0.0126
----------------------------------------
Precision@3:        0.1373 (Average hits within the 3)
Hit Rate@3:         41.18% (Samples with at least 1 hit)
----------------------------------------

>>> TOP-5 ANALYSIS (Forcing the 5 highest probabilities) <<<